<a href="https://colab.research.google.com/github/brevfeed/pyspark/blob/main/notebooks/10-partitioning-shuffle/02-shuffle-mechanics.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

*Part of the [BrevFeed PySpark notebooks](https://github.com/brevfeed/pyspark). Runs on real Spark 4.0 — for the in-browser course see [brevfeed.com/learn/pyspark](https://brevfeed.com/learn/pyspark).*


In [ ]:
# --- Colab setup: run me first (no-op if you already have Spark) --------------
# Installs a JDK + PySpark and fetches the sample data. ~1-2 min on a cold
# Colab runtime; instant on re-run. Safe to run locally too.
import os
import re
import subprocess
import sys
import urllib.request

DATA_URL = "https://raw.githubusercontent.com/brevfeed/pyspark/main/data"
DATA_DIR = "data"
IN_COLAB = "google.colab" in sys.modules


def _java_version():
    """Major version of the JDK on PATH, or 0 if there isn't one."""
    try:
        out = subprocess.run(["java", "-version"], capture_output=True, text=True).stderr
    except (FileNotFoundError, OSError):
        return 0
    m = re.search(r'version "(\d+)', out)
    return int(m.group(1)) if m else 0


# Spark 4 requires Java 17+. Colab ships an older JDK on some images.
if _java_version() < 17:
    if IN_COLAB:
        print("Installing OpenJDK 17 ...")
        subprocess.run(["apt-get", "-qq", "update"], check=False)
        subprocess.run(
            ["apt-get", "-qq", "install", "-y", "openjdk-17-jdk-headless"], check=True
        )
        os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
    else:
        raise SystemExit(
            "Java 17+ is required for Spark 4. Install a JDK 17 and set JAVA_HOME."
        )

try:
    import pyspark  # noqa: F401
except ImportError:
    print("Installing PySpark ...")
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "pyspark==4.0.0"], check=True
    )

# Sample data lives in the repo; pull only what's missing.
os.makedirs(DATA_DIR, exist_ok=True)
for _f in ("orders.csv", "customers.csv", "events.jsonl", "sample_logs.txt"):
    _p = os.path.join(DATA_DIR, _f)
    if not os.path.exists(_p):
        urllib.request.urlretrieve(f"{DATA_URL}/{_f}", _p)

print("Ready. Sample data in ./data:", sorted(os.listdir(DATA_DIR)))


# 10.2 — Shuffle mechanics: catch the shuffle in the act

Count Exchanges in plans, find the actual `.data`/`.index` shuffle
files on your disk, itemize shuffle bytes through the UI's REST
API, and watch map-side partial aggregation earn (and lose) its
discount.

In [ ]:
import os, sys
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.functions import col

spark = (
    SparkSession.builder
    .appName("10.2-shuffle-mechanics")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.sql.adaptive.enabled", "false")
    # point Spark's scratch space somewhere we can go look at it
    .config("spark.local.dir", "spark-local")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

## Exchanges in the plan = shuffles in the job

Narrow ops: zero Exchanges. A groupBy: one, wrapped in
`HashAggregate(partial)` below and `HashAggregate(final)` above —
map-side combine, visible in the plan. A sort-merge join: two.

In [ ]:
df = spark.range(1_000_000, numPartitions=8).withColumn(
    "k", (col("id") % 50).cast("int")).withColumn("v", col("id") * 2)

df.where(col("v") > 100).select("k").explain()          # 0 exchanges
df.groupBy("k").sum("v").explain()                      # 1 — find both HashAggregates
df.alias("a").join(df.alias("b"), "k").explain()        # 2 — one per side

## The shuffle files, on your actual disk

Sort-based shuffle: each map task writes exactly one `.data` file
(rows laid out by destination partition) plus one `.index` file
(byte offsets). 8 map tasks → 8 + 8 files, no matter how many
reduce partitions. Files accumulate per shuffle until the app
cleans them up.

In [ ]:
def shuffle_file_report():
    data, index, total_bytes = 0, 0, 0
    for root, _, files in os.walk("spark-local"):
        for f in files:
            if f.startswith("shuffle_"):
                data += f.endswith(".data")
                index += f.endswith(".index")
                total_bytes += os.path.getsize(os.path.join(root, f))
    print(f"{data} .data + {index} .index files, {total_bytes:,} bytes total")

shuffle_file_report()                      # before: nothing yet
df.groupBy("k").sum("v").collect()         # force one shuffle (8 map tasks)
shuffle_file_report()                      # after: 8 + 8

# a shuffle is a DISK WRITE even though our code never said "write"

## Itemizing the bill: shuffle metrics via the UI's REST API

Everything on the Stages tab is also JSON at
`localhost:4040/api/v1/...` — a taste of scripted diagnosis
(Module 13 leans on this). Stage N writes what stage N+1 reads.

In [ ]:
import urllib.request, json

def recent_stages(n=4):
    app_id = spark.sparkContext.applicationId
    url = (f"http://localhost:4040/api/v1/applications/{app_id}"
           f"/stages?status=complete")
    stages = json.load(urllib.request.urlopen(url))
    for s in sorted(stages, key=lambda s: s["stageId"])[-n:]:
        print(f"stage {s['stageId']:>3}: "
              f"write {s['shuffleWriteBytes']:>12,} B / "
              f"{s['shuffleWriteRecords']:>9,} rec | "
              f"read {s['shuffleReadBytes']:>12,} B / "
              f"{s['shuffleReadRecords']:>9,} rec")

recent_stages()

## The partial-aggregation discount — earned, then lost

`sum` shrinks each map task's million rows to ≤50 partial rows
before anything crosses the wire. `collect_list` can't shrink
anything — every value ships. Same groupBy, ~1000× the shuffle.

In [ ]:
df.groupBy("k").agg(F.sum("v")).collect()
print("sum — partials shrink the shuffle:")
recent_stages(2)

df.groupBy("k").agg(F.collect_list("v")).collect()
print("\ncollect_list — every row crosses the wire:")
recent_stages(2)

## ReusedExchange: the shuffle Spark does once and reads twice

The same aggregated subtree on both sides of a join: one Exchange
runs, the other side reads its files again for free.

In [ ]:
agg = df.groupBy("k").agg(F.sum("v").alias("total"))

(agg.alias("a")
 .join(agg.alias("b"),
       (col("a.total") == col("b.total")) & (col("a.k") < col("b.k")))
 .explain())
# find ReusedExchange in the output — free caching, no cache() call

In [ ]:
import shutil
spark.stop()                                     # releases the shuffle files
shutil.rmtree("spark-local", ignore_errors=True) # self-clean
# (rerun the setup cell if you want to continue experimenting)

## Exercises

1. Before running it, predict the number of Exchanges in:
   `df.distinct()`, `df.orderBy("v")`,
   `df.groupBy("k").count().orderBy("count")`, and a broadcast
   join. Verify with `explain()`.
2. Rerun the shuffle-file demo with `numPartitions=3` on the range.
   Predict the file count first. Then check the byte-offsets story:
   why is the `.index` file always tiny?
3. Using `recent_stages`, measure the shuffle for
   `df.groupBy("k").agg(F.countDistinct("v"))`. It's far bigger
   than plain `count`. Run `explain()` and find the extra Exchange —
   what does Spark expand countDistinct into?
4. The lesson claims write ≈ read for one exchange. Find a case in
   your metrics where a stage's read is much LARGER than the
   previous stage's write, and explain it (hint: a join reads from
   two exchanges).
5. Kill the partial aggregation on purpose: compare shuffle records
   for `groupBy(k).sum(v)` when `k` has 50 distinct values vs when
   `k` is unique per row (`groupBy("id")`). Why does the discount
   disappear even for `sum`?

In [ ]:
# your exercise code here